# Implement your own algorithm: REINFORCE

This notebook is a **skeleton**: the framework plumbing is wired, the
learning update is yours to write. Fill in `train_on_batch` (and whatever
state you need in `setup`) and everything else — collection, logging,
checkpointing, evaluation, the run directory — comes from the Trainer.

Training opens the **interactive Genesis viewer** so you can watch the cars
learn in real time (set `SHOW_VIEWER = False` on a headless machine and use
TensorBoard instead).

In [ ]:
import torch

from deepracer_genesis.algorithms import register_algorithm
from deepracer_genesis.experiment import Algo, FeatureEnvironment, VectorPolicy
from deepracer_genesis.experiment.builder import Builder
from deepracer_genesis.experiment.trainer import Trainer

SHOW_VIEWER = True     # False on headless machines (watch TensorBoard instead)
NUM_ENVS = 64          # keep the viewer readable; bump when headless

## The algorithm

The contract (see `deepracer_genesis/algorithms/protocol.py`):

- `setup(builder)` — build your networks/optimizers. The `Builder` hands you
  ready-made pieces: `builder.actor()` (a stochastic TanhNormal or
  Categorical actor matching the spec), `builder.critic()`,
  `builder.optimizer(module)`, plus `builder.spec` and `builder.sim()`.
- `collect_policy` / `eval_actor` — what the collector runs / what
  deterministic evaluation uses. Usually both are your actor.
- `train_on_batch(data)` — **your update.** `data` is one collector yield,
  a TensorDict of shape `[num_envs, horizon]` with:
  `data["action"]`, `data["action_log_prob"]` (log pi(a|s) at collection
  time), `data["next", "reward"]` `[N, T, 1]`, `data["next", "done"]`,
  and the observations under `data["state"]`. Return a dict of floats —
  they land in TensorBoard/MLflow automatically.
- `observe_env_logs(logs)` / `checkpoint()` — optional hooks.

In [ ]:
@register_algorithm("reinforce")
class REINFORCE:
    """YOUR algorithm. The stubs below run (badly) — make them learn."""

    def setup(self, builder: Builder) -> None:
        self.spec = builder.spec
        self.actor = builder.actor()
        self.optim = builder.optimizer(self.actor)
        self.gamma = builder.spec.algorithm.params.get("gamma", 0.99)

    @property
    def collect_policy(self):
        return self.actor

    @property
    def eval_actor(self):
        return self.actor

    def train_on_batch(self, data) -> dict:
        # data: TensorDict [N, T]; see the cell above for the keys.
        #
        # TODO(you): REINFORCE —
        #   1. returns-to-go G_t from data["next", "reward"] (respect
        #      data["next", "done"]: episodes end mid-batch!)
        #   2. recompute log-probs: dist = self.actor.get_dist(data)
        #      logp = dist.log_prob(data["action"])
        #   3. loss = -(logp * G).mean()   (+ optional baseline)
        #   4. backprop + step
        raise NotImplementedError("write your REINFORCE update here")
        return {"Loss/reinforce": 0.0}

    def observe_env_logs(self, logs: dict) -> None:
        pass

    def checkpoint(self) -> dict:
        return {"actor": self.actor.state_dict()}

## Run it

`Algo(kind="reinforce")` routes the pipeline to your class. We drive the
Builder/Trainer directly (instead of `run()`) so we can switch the viewer
on — a purely visual setting that shouldn't change the run's identity hash.

In [ ]:
spec = (
    FeatureEnvironment(num_envs=NUM_ENVS)
    >> VectorPolicy()
    >> Algo(kind="reinforce", params={"gamma": 0.99})
).build(ablation_group="custom_algos", variant="reinforce",
        total_env_steps=2_000_000, eval_every_steps=500_000)

builder = Builder(spec)
builder.sim(extra_cfg={"show_viewer": SHOW_VIEWER})   # watch it train
record = Trainer(builder, root="runs").fit(force=True)
record.metrics

## Watch the result

```python
from deepracer_genesis.experiment.visualize import rollout_video
rollout_video(spec)
```

When it works, promote the class to a module, keep the
`@register_algorithm("reinforce")` decorator, and any experiment file can
use `Algo(kind="reinforce")` from then on.